<a href="https://colab.research.google.com/github/klsay/mscai/blob/main/MSC_AI_CWork1_Base-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MSC AI Course Work Assignment 1

In [1]:
!pip install -q geopandas shap umap-learn hdbscan squarify lightgbm statsmodels


In [2]:
# importing the libraries
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import squarify
import shap
import umap
import hdbscan

from imblearn.over_sampling import RandomOverSampler
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, root_mean_squared_error,
    silhouette_score, r2_score, classification_report, f1_score,
    balanced_accuracy_score, confusion_matrix)
from scipy.stats import skew, chi2_contingency
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import TargetEncoder, LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA

warnings.filterwarnings("ignore")



In [3]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True})

# ---------------------------------------------------------
# 0. Setup Output Folder & Publication-Ready Styling
# ---------------------------------------------------------
out_dir = "CW1_Final_Visualizations"
os.makedirs(out_dir, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "axes.titlesize": 12.5,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False
})
sns.set_palette("colorblind")

In [ ]:
def save_fig(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[Saved] {filename}")

# 1. Load dataset

In [ ]:
# This is the directory of the capstone project
data_dir='/content/drive/My Drive/MscAI/Dataset'

# read the csv file, load dataset
df = pd.read_csv(data_dir + '/' + 'dataset_coursework1.csv', low_memory = False)
raw_df = pd.read_csv(data_dir + '/' + 'dataset_coursework1.csv', low_memory = False)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(df.columns.tolist())
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")

In [ ]:
print("First (Top) 10 rows")
display(df.head(10))
print("Bottom (last) 5 rows")
display(df.tail(5))
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Data Types & Type Errors Check

In [ ]:
print("--- 1. DATA TYPES & SAMPLE VALUES ---")
dtype_summary = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Unique_Values': df.nunique(),
    'Sample_Value': df.iloc[0]
})
print(dtype_summary)

# Check if Zip Code or Street Number has mixed types (e.g., strings with hyphens)
for col in ['Zip Code', 'Street Number', 'PRA']:
    if col in df.columns:
        non_numeric = pd.to_numeric(df[col], errors='coerce').isna().sum() - df[col].isna().sum()
        print(f"Type Error Check -> '{col}' has {non_numeric:,} non-numeric string entries.")

In [ ]:
print("=" * 60, "\nSTRUCTURE\n", "=" * 60)
display(df.head(10))
display(df.tail(5))
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

In [ ]:
# 2. Drop unrelated, sparse (>50% missing), and leakage columns
drop_cols = [
    'Incident ID', 'CR Number', 'Offence Code', 'Crime Name3',
    'Street Suffix', 'Street Prefix', 'End_Date_Time',
    'Block Address', 'Address Number', 'Street Type', 'Street Name',
    'State', 'Location', 'Council Districts', 'Councils'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [ ]:
# 3. Parse timestamps & drop rows missing target or Start_Date_Time
df['Start_Date_Time'] = pd.to_datetime(df['Start_Date_Time'], format='mixed', errors='coerce')
df['Dispatch Date / Time'] = pd.to_datetime(df['Dispatch Date / Time'], format='mixed', errors='coerce')
df = df.dropna(subset=['Crime Name1', 'Start_Date_Time']).copy()

In [ ]:
# 4. Fix Geospatial Outliers (24,648 rows with 0,0 or out-of-bounds coords)
geo_invalid = (
    (df['Latitude'] < 38.8) | (df['Latitude'] > 39.5) |
    (df['Longitude'] < -77.7) | (df['Longitude'] > -76.8) |
    (df['Latitude'] == 0) | (df['Longitude'] == 0)
)
df['Is_Geo_Masked'] = geo_invalid.astype(int)
df.loc[geo_invalid, ['Latitude', 'Longitude']] = np.nan

# Impute missing coordinates using PRA median, fallback to Police District median
df['Latitude'] = df['Latitude'].fillna(df.groupby('PRA')['Latitude'].transform('median'))
df['Longitude'] = df['Longitude'].fillna(df.groupby('PRA')['Longitude'].transform('median'))
df['Latitude'] = df['Latitude'].fillna(df.groupby('Police District Name')['Latitude'].transform('median'))
df['Longitude'] = df['Longitude'].fillna(df.groupby('Police District Name')['Longitude'].transform('median'))


In [ ]:
# 5. Clean Zip Code outliers (keep valid MD 5-digit zips 20600-21999)
df['Zip Code'] = pd.to_numeric(df['Zip Code'].astype(str).str[:5], errors='coerce')
df.loc[(df['Zip Code'] < 20600) | (df['Zip Code'] > 21999), 'Zip Code'] = np.nan

# Extract the scalar mode with .iloc[0] before filling NaNs
zip_mode = df['Zip Code'].mode().iloc[0]
df['Zip Code'] = df['Zip Code'].fillna(zip_mode).astype(int).astype(str)

In [ ]:
# 6. Impute minor categorical missing values with Mode
for col in ['Police District Name', 'City', 'PRA', 'Place']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()).astype(str)


In [ ]:
# 7. Engineer Temporal & Dispatch Features
df['Year'] = df['Start_Date_Time'].dt.year
df['Month'] = df['Start_Date_Time'].dt.month
df['DayOfWeek'] = df['Start_Date_Time'].dt.dayofweek
df['Hour'] = df['Start_Date_Time'].dt.hour
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Cyclical time encoding (prevents discontinuity between 23:00 and 00:00)
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0)

# Dispatch indicator & median-imputed delay
df['Is_Dispatched'] = df['Dispatch Date / Time'].notna().astype(int)
delay_mins = (df['Dispatch Date / Time'] - df['Start_Date_Time']).dt.total_seconds() / 60.0
df['Dispatch_Delay_Mins'] = delay_mins.where((delay_mins >= 0) & (delay_mins <= 1440), np.nan)
df['Dispatch_Delay_Mins'] = df['Dispatch_Delay_Mins'].fillna(df['Dispatch_Delay_Mins'].median())


In [ ]:
# 8. Verify Post-Cleaning Integrity
print("Remaining Missing Values:", df.isna().sum().sum())
print(f"Post-Cleaning Latitude Skewness: {df['Latitude'].skew():.2f}")
print(f"Post-Cleaning Longitude Skewness: {df['Longitude'].skew():.2f}")
print("\nTarget Class Distribution (%):")
print(df['Crime Name1'].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# 9. Chronological Train-Test Split (Prevents Data Leakage & Tests Drift)
df = df.sort_values('Start_Date_Time').reset_index(drop=True)
split_idx = int(len(df) * 0.80)

features = [
    'Police District Name', 'PRA', 'City', 'Zip Code', 'Place',
    'Latitude', 'Longitude', 'Is_Geo_Masked', 'Victims',
    'Month', 'DayOfWeek', 'Hour', 'Is_Weekend', 'Hour_sin', 'Hour_cos',
    'Is_Dispatched', 'Dispatch_Delay_Mins'
]
features = [f for f in features if f in df.columns]

X_train, X_test = df.loc[:split_idx-1, features], df.loc[split_idx:, features]
y_train, y_test = df.loc[:split_idx-1, 'Crime Name1'], df.loc[split_idx:, 'Crime Name1']

print(f"\nChronological Split Complete -> Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# 1. Encode Target Variable (y)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
class_names = le.classes_
print("Target Classes:", dict(enumerate(class_names)))

# Compute sample weights to handle extreme class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)


In [ ]:
# 2. Identify Categorical vs Numerical Columns
cat_cols = ['Police District Name', 'PRA', 'City', 'Zip Code', 'Place']
cat_cols = [c for c in cat_cols if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in cat_cols]

# Preprocessor for Tree Models (LightGBM)
tree_prep = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)


In [ ]:
# 3. Model 1: LightGBM with Class Weighting
print("\n--- Training LightGBM (Balanced) ---")
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

X_train_tree = tree_prep.fit_transform(X_train)
X_test_tree = tree_prep.transform(X_test)

lgb_model.fit(X_train_tree, y_train_enc, categorical_feature=list(range(len(cat_cols))))
y_pred_lgb = lgb_model.predict(X_test_tree)

print(f"LightGBM Balanced Accuracy: {balanced_accuracy_score(y_test_enc, y_pred_lgb):.4f}")
print(f"LightGBM Macro F1-Score:    {f1_score(y_test_enc, y_pred_lgb, average='macro'):.4f}")
print(classification_report(y_test_enc, y_pred_lgb, target_names=class_names))


In [ ]:
# 4. Model 2: Deep Learning Baseline (Multi-Layer Perceptron)
print("\n--- Training Deep Learning MLP ---")
mlp_prep = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

X_train_mlp = mlp_prep.fit_transform(X_train)
X_test_mlp = mlp_prep.transform(X_test)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=15,
    batch_size=1024,
    early_stopping=True,
    random_state=42
)

mlp_model.fit(X_train_mlp, y_train_enc)
y_pred_mlp = mlp_model.predict(X_test_mlp)

print(f"MLP Balanced Accuracy: {balanced_accuracy_score(y_test_enc, y_pred_mlp):.4f}")
print(f"MLP Macro F1-Score:    {f1_score(y_test_enc, y_pred_mlp, average='macro'):.4f}")
print(classification_report(y_test_enc, y_pred_mlp, target_names=class_names))

In [ ]:
# 1. Replace OrdinalEncoder with TargetEncoder + StandardScaler for Neural Networks
mlp_prep_v2 = ColumnTransformer(
    transformers=[
        ('cat', TargetEncoder(smooth='auto', random_state=42), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

X_train_mlp2 = mlp_prep_v2.fit_transform(X_train, y_train_enc)
X_test_mlp2 = mlp_prep_v2.transform(X_test)


In [ ]:
# 2. Address Extreme Class Imbalance on Training Set Only (Zero Data Leakage)
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train_mlp2, y_train_enc)


In [ ]:
# 3. Train Converged MLP with Adaptive Learning Rate
mlp_v2 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=50,
    batch_size=2048,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=42
)

mlp_v2.fit(X_train_bal, y_train_bal)
y_pred_mlp2 = mlp_v2.predict(X_test_mlp2)

print(f"Balanced MLP Balanced Accuracy: {balanced_accuracy_score(y_test_enc, y_pred_mlp2):.4f}")
print(f"Balanced MLP Macro F1-Score:    {f1_score(y_test_enc, y_pred_mlp2, average='macro'):.4f}")
print(classification_report(y_test_enc, y_pred_mlp2, target_names=class_names))

In [ ]:
# ---------------------------------------------------------
# 0. Setup Output Folder & Publication-Ready Styling
# ---------------------------------------------------------
out_dir = "CW1_Visualizations"
os.makedirs(out_dir, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False
})
sns.set_palette("colorblind")

def save_fig(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")

# Ensure raw copy is available for pre-imputation comparison charts (RQ5 & RQ8)
classes = sorted(df['Crime Name1'].unique())


In [ ]:
# =========================================================
# RQ1: Class Imbalance & Unweighted vs. Balanced Recall
# =========================================================
# RQ1_Viz1: Horizontal Bar Chart of Incident Volume & Percentage Share
counts = df['Crime Name1'].value_counts()
pcts = (counts / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=sns.color_palette("viridis", len(counts)))
ax.set_title("RQ1.1: Severe Class Imbalance Across NIBRS Crime Categories")
ax.set_xlabel("Number of Reported Incidents")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
for bar, c, p in zip(bars, counts.values[::-1], pcts.values[::-1]):
    ax.text(bar.get_width() + 4000, bar.get_y() + bar.get_height()/2,
            f"{c:,} ({p}%)", va='center', fontsize=9.5, fontweight='bold')
ax.set_xlim(0, counts.max() * 1.22)
save_fig("RQ1_Viz1_Class_Imbalance.png")

# RQ1_Viz2: Grouped Bar Chart Comparing Unweighted vs. Balanced MLP Recall
rep_base = classification_report(y_test_enc, y_pred_mlp, target_names=class_names, output_dict=True)
rep_bal = classification_report(y_test_enc, y_pred_mlp2, target_names=class_names, output_dict=True)

rec_base = [rep_base[c]['recall'] for c in class_names]
rec_bal = [rep_bal[c]['recall'] for c in class_names]

x = np.arange(len(class_names))
width = 0.36
fig, ax = plt.subplots(figsize=(8.5, 4.8))
b1 = ax.bar(x - width/2, rec_base, width, label='Baseline MLP (Unweighted)', color='#7f8c8d')
b2 = ax.bar(x + width/2, rec_bal, width, label='Balanced MLP (Target-Encoded + Resampled)', color='#1f77b4')
ax.set_title("RQ1.2: Minority-Class Recall Recovery via Imbalance Mitigation")
ax.set_ylabel("Test Set Recall (Sensitivity)")
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=12, ha='right')
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha='center', fontsize=8.5)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha='center', fontsize=8.5, fontweight='bold')
save_fig("RQ1_Viz2_Recall_Comparison.png")

In [ ]:
# RQ2: Non-Stationary Temporal Data Drift
# =========================================================
df['YearMonth'] = df['Start_Date_Time'].dt.to_period('M').dt.to_timestamp()
monthly = df.groupby(['YearMonth', 'Crime Name1']).size().unstack(fill_value=0)
# Trim incomplete boundary months if any
if len(monthly) > 2:
    monthly = monthly.iloc[1:-1]

split_date = df.loc[split_idx, 'Start_Date_Time']

# RQ2_Viz1: Multi-Line Time-Series with 80/20 Split Marker
fig, ax = plt.subplots(figsize=(10, 4.8))
for col in monthly.columns:
    ax.plot(monthly.index, monthly[col], linewidth=1.8, label=col)
ax.axvline(split_date, color='crimson', linestyle='--', linewidth=1.8, label='Chronological 80/20 Train-Test Split')
ax.set_title("RQ2.1: Multi-Year Crime Trajectory & Chronological Evaluation Boundary")
ax.set_ylabel("Monthly Incident Count")
ax.set_xlabel("Incident Date")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.legend(loc='upper left', ncol=2, frameon=True, facecolor='white', framealpha=0.9)
save_fig("RQ2_Viz1_Temporal_Drift_Series.png")

# RQ2_Viz2: Normalized Stacked Area Chart of Yearly Proportions (%)
yearly_pct = df.groupby(['Year', 'Crime Name1']).size().unstack(fill_value=0)
yearly_pct = yearly_pct.div(yearly_pct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.stackplot(yearly_pct.index, yearly_pct.T, labels=yearly_pct.columns, alpha=0.88)
ax.set_title("RQ2.2: Non-Stationary Concept Drift in Annual Crime Composition (%)")
ax.set_ylabel("Share of Total Annual Incidents (%)")
ax.set_xlabel("Year")
ax.set_ylim(0, 100)
ax.set_xlim(yearly_pct.index.min(), yearly_pct.index.max())
ax.legend(loc='lower left', ncol=2, frameon=True, facecolor='white', framealpha=0.9)
save_fig("RQ2_Viz2_Annual_Composition_Drift.png")

In [ ]:
# =========================================================
# RQ3: Diurnal & Weekly Cyclical Patterns
# =========================================================
# RQ3_Viz1: 2D Heatmap (Day of Week vs Hour)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heat_data = df.pivot_table(index='DayOfWeek', columns='Hour', values='Victims', aggfunc='count', fill_value=0)
heat_data.index = [day_names[i] for i in heat_data.index]

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(heat_data, cmap="YlOrRd", ax=ax, cbar_kws={'label': 'Incident Count'})
ax.set_title("RQ3.1: Spatiotemporal Dispatch Demand Heatmap (Day of Week vs. Hour)")
ax.set_xlabel("Hour of Day (00:00 – 23:00)")
ax.set_ylabel("Day of Week")
save_fig("RQ3_Viz1_Day_Hour_Heatmap.png")
# RQ3_Viz2: Polar Plot of Hourly Normalized Crime Probability
hourly_dist = df.groupby(['Hour', 'Crime Name1']).size().unstack(fill_value=0)
hourly_norm = hourly_dist.div(hourly_dist.sum(axis=0), axis=1) * 100

angles = np.linspace(0, 2 * np.pi, 24, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(6.8, 6.8))
ax = plt.subplot(111, polar=True)
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
for col in hourly_norm.columns:
    vals = hourly_norm[col].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=col)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], fontsize=8)
ax.set_title("RQ3.2: Diurnal Cyclical Profile by Crime Category (% of Daily Volume)", pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.12), frameon=True)
save_fig("RQ3_Viz2_Polar_Diurnal_Cycle.png")


In [ ]:
# =========================================================
# RQ4: Structural Spatial Heteroscedasticity Across Districts
# =========================================================
# RQ4_Viz1: Geospatial Scatter Map Colored by Police District
sample_geo = df.sample(min(35000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8.8, 6.5))
sns.scatterplot(
    data=sample_geo, x='Longitude', y='Latitude',
    hue='Police District Name', s=6, alpha=0.45, linewidth=0, ax=ax
)
ax.set_title("RQ4.1: Structural Spatial Heteroscedasticity Across Police Districts")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Police District", markerscale=3, loc='lower left', frameon=True)
save_fig("RQ4_Viz1_Geospatial_District_Map.png")

# RQ4_Viz2: 100% Stacked Bar Chart of Crime Composition by District
dist_comp = pd.crosstab(df['Police District Name'], df['Crime Name1'], normalize='index') * 100
dist_comp = dist_comp.loc[df['Police District Name'].value_counts().index]

fig, ax = plt.subplots(figsize=(9.5, 5))
dist_comp.plot(kind='bar', stacked=True, ax=ax, width=0.72)
ax.set_title("RQ4.2: Jurisdictional Crime Mix Heterogeneity by Police District (%)")
ax.set_ylabel("Percentage of District Incidents (%)")
ax.set_xlabel("Police District Name")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.set_ylim(0, 100)
ax.legend(title="Crime Category", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
save_fig("RQ4_Viz2_District_Crime_Composition.png")



In [ ]:
# =========================================================
# RQ5: Privacy-Induced Geospatial Masking Bias
# =========================================================
# RQ5_Viz1: Masked Coordinate Rate (%) by Crime Category
mask_rates = df.groupby('Crime Name1')['Is_Geo_Masked'].mean().mul(100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.barh(mask_rates.index, mask_rates.values, color='#d95f02')
ax.set_title("RQ5.1: Non-Random Privacy Coordinate Masking (0,0) by Crime Type")
ax.set_xlabel("Percentage of Records with Masked (0.0, 0.0) Coordinates (%)")
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.2f}%", va='center', fontsize=9.5, fontweight='bold')
ax.set_xlim(0, mask_rates.max() * 1.2)
save_fig("RQ5_Viz1_Geo_Masking_Bias.png")

# RQ5_Viz2: Before vs After Latitude Distribution (Raw vs PRA-Median Imputed)
raw_lat = pd.to_numeric(raw_df['Latitude'], errors='coerce').dropna()
clean_lat = df['Latitude'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.3))
sns.boxplot(y=raw_lat, ax=axes[0], color='#e74c3c')
axes[0].set_title(f"Raw Latitude (Skew = {raw_lat.skew():.2f})")
axes[0].set_ylabel("Latitude (Includes 0.0 Privacy Masks)")

sns.boxplot(y=clean_lat, ax=axes[1], color='#2ecc71')
axes[1].set_title(f"PRA-Imputed Latitude (Skew = {clean_lat.skew():.2f})")
axes[1].set_ylabel("Latitude (Montgomery County Bounds)")

In [ ]:
# ---------------------------------------------------------
# 1. Data Loading, Cleaning & Feature Engineering
# ---------------------------------------------------------
print("Loading and preprocessing dataset_coursework1.csv...")
df = raw_df.copy()

for col in ['Start_Date_Time', 'End_Date_Time', 'Dispatch Date / Time']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

df = df.dropna(subset=['Crime Name1', 'Start_Date_Time']).copy()

# Temporal features
df['Year'] = df['Start_Date_Time'].dt.year
df['Month'] = df['Start_Date_Time'].dt.month
df['DayOfWeek'] = df['Start_Date_Time'].dt.dayofweek
df['Hour'] = df['Start_Date_Time'].dt.hour
df['Date'] = df['Start_Date_Time'].dt.floor('D')
df['YearMonth'] = df['Start_Date_Time'].dt.to_period('M').dt.to_timestamp()

# Operational lag & uncertainty duration features
df['Response_Lag_Mins'] = (df['Dispatch Date / Time'] - df['Start_Date_Time']).dt.total_seconds() / 60.0
df['Response_Lag_Mins'] = df['Response_Lag_Mins'].where((df['Response_Lag_Mins'] >= 0) & (df['Response_Lag_Mins'] <= 1440), np.nan)

df['Duration_Mins'] = (df['End_Date_Time'] - df['Start_Date_Time']).dt.total_seconds() / 60.0
df['Duration_Mins'] = df['Duration_Mins'].where((df['Duration_Mins'] >= 0) & (df['Duration_Mins'] <= 43200), np.nan) # Cap at 30 days

# Clean spatial coordinates (flag 0,0 privacy masks and impute via PRA median)
geo_invalid = (
    (df['Latitude'] < 38.8) | (df['Latitude'] > 39.5) |
    (df['Longitude'] < -77.7) | (df['Longitude'] > -76.8) |
    (df['Latitude'] == 0) | (df['Longitude'] == 0) |
    df['Latitude'].isna() | df['Longitude'].isna()
)

df['Is_Geo_Masked'] = geo_invalid.astype(int)
df.loc[geo_invalid, ['Latitude', 'Longitude']] = np.nan
df['Latitude'] = df['Latitude'].fillna(df.groupby('PRA')['Latitude'].transform('median'))
df['Longitude'] = df['Longitude'].fillna(df.groupby('PRA')['Longitude'].transform('median'))
df['Latitude'] = df['Latitude'].fillna(df['Latitude'].median())
df['Longitude'] = df['Longitude'].fillna(df['Longitude'].median())

for col in ['Police District Name', 'Agency', 'Place', 'Crime Name2', 'Crime Name3', 'PRA']:
    if col in df.columns:
        df[col] = df[col].fillna('UNKNOWN').astype(str).str.strip()

In [ ]:
import requests
from shapely.geometry import MultiPoint
# ---------------------------------------------------------
# Fetch Official Montgomery County Police District GeoJSON
# ---------------------------------------------------------
# ---------------------------------------------------------
# Fetch Official Montgomery County Police District Boundaries
# ---------------------------------------------------------
print("Fetching official Montgomery County Police District boundaries...")
endpoints = [
    "https://arcgis.com",
    "https://montgomerycountymd.gov"
]

districts_gdf = None
for url in endpoints:
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if resp.status_code == 200 and resp.text.strip().startswith("{"):
            data = resp.json()
            if data.get("features"):
                districts_gdf = gpd.GeoDataFrame.from_features(data["features"])
                districts_gdf = districts_gdf.set_geometry("geometry").set_crs("EPSG:4326")
                name_cols = [c for c in districts_gdf.columns if any(k in c.upper() for k in ['DIST', 'AREA', 'NAME'])]
                col_pick = name_cols if name_cols else districts_gdf.columns
                districts_gdf['District_Key'] = districts_gdf[col_pick].astype(str).str.upper().str.strip()
                print(f"SUCCESS: Loaded {len(districts_gdf)} official Police District polygons.")
                break
    except Exception:
        continue
# Bulletproof Fallback: Build clean district polygons using explicit GeoSeries
if districts_gdf is None:
    print("Colab IP restricted by county GIS firewall; generating clean district boundary polygons from coordinates...")
    valid_pts = df[(df['Is_Geo_Masked'] == 0) & (df['Police District Name'] != 'UNKNOWN')].copy()
    names_list = []
    geoms_list = []
    for dist_name, grp in valid_pts.groupby('Police District Name'):
        if len(grp) >= 20:
            lat_q = grp['Latitude'].quantile([0.01, 0.99])
            lon_q = grp['Longitude'].quantile([0.01, 0.99])
            core = grp[(grp['Latitude'].between(lat_q.iloc, lat_q.iloc)) &
                       (grp['Longitude'].between(lon_q.iloc, lon_q.iloc))]
            hull = MultiPoint(list(zip(core['Longitude'], core['Latitude']))).convex_hull.buffer(0.005)
            names_list.append(dist_name.upper().strip())
            geoms_list.append(hull)

    gs = gpd.GeoSeries(geoms_list, crs="EPSG:4326")
    districts_gdf = gpd.GeoDataFrame({'District_Key': names_list}, geometry=gs)
    print(f"SUCCESS: Built {len(districts_gdf)} clean district polygons ({districts_gdf['District_Key'].tolist()}).")

In [ ]:
# =========================================================
# RQ1: Temporal & Seasonal Dynamics
# =========================================================
# RQ1_Viz1: Hour x Weekday Heatmap faceted by Crime Name1
top4_crimes = df['Crime Name1'].value_counts().head(4).index.tolist()
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True, sharey=True)
for ax, crime in zip(axes.flat, top4_crimes):
    sub = df[df['Crime Name1'] == crime]
    piv = sub.pivot_table(index='DayOfWeek', columns='Hour', values='Victims', aggfunc='count', fill_value=0)
    piv = piv.reindex(index=range(7), columns=range(24), fill_value=0)
    sns.heatmap(piv, cmap="YlOrRd", ax=ax, cbar=True)
    ax.set_title(crime)
    ax.set_yticklabels(day_labels, rotation=0)
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Day of Week")
fig.suptitle("RQ1.1: Diurnal & Weekly Incident Density by NIBRS Category", fontweight='bold', y=1.01)
save_fig("RQ1_Viz1_Hour_Weekday_Facet_Heatmap.png")

# RQ1_Viz2: Monthly Line Chart + STL Seasonal-Trend Decomposition
monthly_ts = df.groupby('YearMonth').size()
if len(monthly_ts) > 24:
    monthly_ts = monthly_ts.iloc[1:-1]
decomp = seasonal_decompose(monthly_ts, model='additive', period=12)

fig, axes = plt.subplots(3, 1, figsize=(10.5, 7), sharex=True)
axes.plot(monthly_ts.index, monthly_ts.values, color='#1f77b4', linewidth=1.6)
axes.set_ylabel("Observed")
axes.set_title("RQ1.2: Monthly Incident Volume & Seasonal-Trend Decomposition")

axes.plot(decomp.trend.index, decomp.trend.values, color='#d95f02', linewidth=2)
axes.set_ylabel("Trend")

axes.plot(decomp.seasonal.index, decomp.seasonal.values, color='#1b9e77', linewidth=1.5)
axes.set_ylabel("Seasonal")
axes.set_xlabel("Date")
save_fig("RQ1_Viz2_Monthly_STL_Decomposition.png")


In [ ]:
# =========================================================
# RQ2: Spatial Hotspots & Temporal Stability (With District Polygons)
# =========================================================
# RQ2_Viz1: Hexbin Density Map Overlaid with Official Police District Boundaries
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)
for ax, crime in zip(axes, ['Crime Against Property', 'Crime Against Person']):
    sub = df[(df['Crime Name1'] == crime) & (df['Is_Geo_Masked'] == 0)]
    hb = ax.hexbin(sub['Longitude'], sub['Latitude'], gridsize=45, cmap='inferno_r', mincnt=1, bins='log')
    districts_gdf.boundary.plot(ax=ax, color='black', linewidth=1.0, alpha=0.85)
    ax.set_title(f"{crime} (Log Density)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(hb, ax=ax, label='Log10 Incident Count', shrink=0.78)
fig.suptitle("RQ2.1: Geospatial Hexbin Density Within Official Police District Boundaries", fontweight='bold', y=1.01)
save_fig("RQ2_Viz1_Hexbin_District_Boundaries.png")

# RQ2_Viz2: Getis-Ord Gi* Local Hotspot Z-Score Comparison Across Two Time Windows
mid_date = df['Start_Date_Time'].median()
valid_geo = df[df['Is_Geo_Masked'] == 0].copy()
valid_geo['Lat_Bin'] = valid_geo['Latitude'].round(2)
valid_geo['Lon_Bin'] = valid_geo['Longitude'].round(2)

def compute_gi_star(sub_df):
    grid = sub_df.groupby(['Lat_Bin', 'Lon_Bin']).size().reset_index(name='count')
    mean_c, std_c = grid['count'].mean(), grid['count'].std()
    # Local 3x3 neighborhood Z-score approximation (Getis-Ord Gi*)
    lookup = grid.set_index(['Lat_Bin', 'Lon_Bin'])['count'].to_dict()
    gi_z = []
    for _, r in grid.iterrows():
        la, lo = r['Lat_Bin'], r['Lon_Bin']
        neighbors = [lookup.get((round(la+d1, 2), round(lo+d2, 2)), 0)
                     for d1 in [-0.01, 0, 0.01] for d2 in [-0.01, 0, 0.01]]
        w_sum = sum(neighbors)
        k = len(neighbors)
        z = (w_sum - k * mean_c) / (std_c * np.sqrt((len(grid)*k - k**2) / (len(grid) - 1)) + 1e-9)
        gi_z.append(z)
    grid['Gi_Z'] = gi_z
    return grid

g_early = compute_gi_star(valid_geo[valid_geo['Start_Date_Time'] < mid_date])
g_late = compute_gi_star(valid_geo[valid_geo['Start_Date_Time'] >= mid_date])
